# CircuitVision: train the component detector on Colab
Runtime → Change runtime type → **T4 GPU**. Run the cells top to bottom.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/sudo-pranshu/CircuitVision.git
%cd CircuitVision
!pip install -q -e ".[detect]"

## Dataset (CGHD, ~4.4 GB, CC BY 3.0)
Cached on Drive after the first download.

In [ ]:
import os
ZIP = '/content/drive/MyDrive/cghd-zenodo-14.zip'
if not os.path.exists(ZIP):
    !wget -q --show-progress -O {ZIP} 'https://zenodo.org/records/14042961/files/cghd-zenodo-14.zip?download=1'
!unzip -q -o {ZIP} -d /content/cghd

In [ ]:
import glob, os
root = os.path.dirname(sorted(glob.glob('/content/cghd/**/drafter_1', recursive=True))[0])
print(root)
!python -m circuitvision.cghd_to_yolo --cghd {root} --out data/yolo

## Train
Try `--quick` first to check everything runs end to end.

In [ ]:
!python -m circuitvision.train --data data/yolo/dataset.yaml --epochs 100 --imgsz 1024 --batch 16

In [ ]:
import glob, shutil
best = sorted(glob.glob('runs/detect*/weights/best.pt'))[-1]
shutil.copy(best, '/content/drive/MyDrive/circuitvision_best.pt')
print('saved', best)

## Evaluate on the test drafter

In [ ]:
!apt-get -qq install ngspice > /dev/null
import glob
d = sorted(glob.glob(f'{root}/drafter_12'))[0]
!python -m circuitvision.evaluate --images {d}/images --annotations {d}/annotations --weights {best} --csv eval_test.csv